In [8]:
import numpy as np
import rasterio
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
import json

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land"
)

quarter_dir = ROOT / "data" / "quarterly_tiffs"
threshold_dir = ROOT / "data" / "percentile_tiffs"
output_dir = ROOT / "data" / "monthly_heatday_scores"

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/assets/district.geojson"
)

output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD DISTRICT GEOMETRY
# =========================================================

gdf = gpd.read_file(geojson_path)
geoms = json.loads(gdf.to_json())["features"]
geoms = [f["geometry"] for f in geoms]

# =========================================================
# REFERENCE GRID (for consistency)
# =========================================================

ref_file = quarter_dir / "HI_2023_Q1.tif"

with rasterio.open(ref_file) as src:
    ref_meta = src.meta.copy()

# =========================================================
# LOAD SINGLE BAND RASTER
# =========================================================

def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1)

# =========================================================
# MONTHLY LOOP
# =========================================================

years = [2023, 2024]

for year in years:

    print(f"\n================ YEAR {year} ================")

    for month in range(1, 13):

        print(f"\nProcessing {year}-{month:02d}")

        # -------------------------------------------------
        # SEASON MAPPING
        # -------------------------------------------------

        if month in [1, 2, 3]:
            season, q = "JFM", "Q1"
        elif month in [4, 5, 6]:
            season, q = "AMJ", "Q2"
        elif month in [7, 8, 9]:
            season, q = "JAS", "Q3"
        else:
            season, q = "OND", "Q4"

        hi_path = quarter_dir / f"HI_{year}_{q}.tif"

        if not hi_path.exists():
            print("SKIP: HI missing")
            continue

        # -------------------------------------------------
        # LOAD THRESHOLDS
        # -------------------------------------------------

        try:
            p80 = load_raster(threshold_dir / f"{season}_P80_1990_2023.tif")
            p88 = load_raster(threshold_dir / f"{season}_P88_1990_2023.tif")
            p95 = load_raster(threshold_dir / f"{season}_P95_1990_2023.tif")
            p99 = load_raster(threshold_dir / f"{season}_P99_1990_2023.tif")
        except Exception as e:
            print(f"SKIP: missing thresholds for {season}")
            continue

        # -------------------------------------------------
        # LOAD HI STACK (days, rows, cols)
        # -------------------------------------------------

        with rasterio.open(hi_path) as src:
            hi = src.read()
            meta = src.meta.copy()

        days, rows, cols = hi.shape

        # -------------------------------------------------
        # MONTHLY HEATDAY SCORE
        # -------------------------------------------------

        monthly = np.zeros((rows, cols), dtype=np.float32)

        for d in range(days):

            h = hi[d]

            score = np.zeros((rows, cols), dtype=np.uint8)

            score[(h >= p80) & (h < p88)] = 1
            score[(h >= p88) & (h < p95)] = 2
            score[(h >= p95) & (h < p99)] = 3
            score[h >= p99] = 4

            monthly += score

        # =========================================================
        # WRITE TEMP RASTER
        # =========================================================

        tmp_file = output_dir / f"_tmp_{year}_{month:02d}.tif"

        meta.update({
            "count": 1,
            "dtype": "float32",
            "nodata": np.nan
        })

        with rasterio.open(tmp_file, "w", **meta) as dst:
            dst.write(monthly, 1)

        # =========================================================
        # PROPER CLIPPING (NO BLACK BOX FIX)
        # =========================================================

        with rasterio.open(tmp_file) as src:

            out_image, out_transform = mask(
                src,
                geoms,
                crop=True,
                filled=True,
                nodata=np.nan
            )

            out_image = out_image.astype("float32")
            out_image[out_image == 0] = np.nan

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "count": 1,
                "nodata": np.nan
            })

        # =========================================================
        # FINAL SAVE
        # =========================================================

        final_file = output_dir / f"HEATDAY_{year}_{month:02d}.tif"

        with rasterio.open(final_file, "w", **out_meta) as dst:
            dst.write(out_image[0], 1)

        tmp_file.unlink()

        print("Saved:", final_file)


================ YEAR 2023 ================

Processing 2023-01
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_01.tif

Processing 2023-02
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_02.tif

Processing 2023-03
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_03.tif

Processing 2023-04
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_04.tif

Processing 2023-05
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_05.tif

Processing 2023-06
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_06.tif

Processing 2023-07
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extracto